In [10]:
from pathlib import Path
import pandas as pd
from Bio.PDB import MMCIFParser
from Bio.PDB.DSSP import DSSP


STRUCT_DIR = Path("data/raw/structures_cif")
OUT_CSV = Path("data/processed/dssp_rsa_all.csv")
OUT_FAIL = Path("data/processed/dssp_failed.csv")

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)


In [11]:
MAX_ASA = {
    "A": 121.0, "R": 265.0, "N": 187.0, "D": 187.0, "C": 148.0,
    "Q": 214.0, "E": 214.0, "G":  97.0, "H": 216.0, "I": 195.0,
    "L": 191.0, "K": 230.0, "M": 203.0, "F": 228.0, "P": 154.0,
    "S": 143.0, "T": 163.0, "W": 264.0, "Y": 255.0, "V": 165.0
}

def asa_to_rsa(aa: str, asa: float):
    """Return RSA in [0,1] when possible, else None."""
    if aa not in MAX_ASA:
        return None
    rsa = asa / MAX_ASA[aa]
    # occasionally can slightly exceed 1 depending on conventions; clamp
    return float(max(0.0, min(1.0, rsa)))

In [12]:
parser = MMCIFParser(QUIET=True)

def dssp_from_cif(cif_path: Path):
    pdb_id = cif_path.stem.lower()
    structure = parser.get_structure(pdb_id, str(cif_path))
    model = structure[0]  # use first model
    dssp = DSSP(model, str(cif_path), dssp="mkdssp")
    return pdb_id, dssp

In [26]:
cif_files = sorted(STRUCT_DIR.glob("*.cif"))
print("mmCIF files found:", len(cif_files))

rows = []
failed = []

for i, cif_path in enumerate(cif_files, 1):
    pdb_id = cif_path.stem.lower()
    try:
        pdb_id, dssp = dssp_from_cif(cif_path)
    except Exception as e:
        failed.append({"pdb_id": pdb_id, "file": str(cif_path), "error": type(e).__name__})
        continue

    # dssp keys look like: (chain_id, (' ', resseq, icode))
    for (chain_id, res_id) in dssp.keys():
        aa = dssp[(chain_id, res_id)][1]   # one-letter AA (or 'X')
        ss = dssp[(chain_id, res_id)][2]   # secondary structure code
        rsa = dssp[(chain_id, res_id)][3]

        hetflag, resseq, icode = res_id

        rows.append({
            "pdb_id": pdb_id,
            "chain_id": str(chain_id),
            "resseq": int(resseq),
            "icode": (icode.strip() if isinstance(icode, str) else ""),
            "aa": aa,
            "ss": ss,
            "rsa": rsa,
        })

    if i % 100 == 0 or i == len(cif_files):
        print(f"[{i}/{len(cif_files)}] rows={len(rows)} failed_structures={len(failed)}")

labels_df = pd.DataFrame(rows)
fails_df = pd.DataFrame(failed)

print("Final rows:", len(labels_df))
print("Failed structures:", len(fails_df))
labels_df.head()

mmCIF files found: 4314


/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is lower than requested, this may cause validation errors
Loaded dictionary does not match name=mmcif_pdbx.dic and version=5.406

  warnings.warn(err)
/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is lower than requested, this may cause validation errors
Loaded dictionary does not match name=mmcif_pdbx.dic and version=5.406

  warnings.warn(err)


[100/4314] rows=41807 failed_structures=0


/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is lower than requested, this may cause validation errors
Loaded dictionary does not match name=mmcif_pdbx.dic and version=5.409

  warnings.warn(err)


[200/4314] rows=87192 failed_structures=0


...

/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is lower than requested, this may cause validation errors
Loaded dictionary does not match name=mmcif_pdbx.dic and version=5.407

  warnings.warn(err)
/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is lower than requested, this may cause validation errors
Loaded dictionary does not match name=mmcif_pdbx.dic and version=5.408

  warnings.warn(err)
/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is lower than requested, this may cause validation errors
Loaded dictionary does not match name=mmcif_pdbx.dic and version=5.408

  warnings.warn(err)
/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is 

[4300/4314] rows=2015951 failed_structures=40


/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is lower than requested, this may cause validation errors
Loaded dictionary does not match name=mmcif_pdbx.dic and version=5.407

  warnings.warn(err)
/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is lower than requested, this may cause validation errors
Loaded dictionary does not match name=mmcif_pdbx.dic and version=5.407

  warnings.warn(err)
/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is lower than requested, this may cause validation errors
Loaded dictionary does not match name=mmcif_pdbx.dic and version=5.410

  warnings.warn(err)
/opt/anaconda3/envs/miniproject/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The version in dictionary mmcif_pdbx.dic is 

[4314/4314] rows=2023533 failed_structures=40
Final rows: 2023533
Failed structures: 40


,pdb_id,chain_id,resseq,icode,aa,ss,rsa
0,12as,A,4,,A,-,0.90566
1,12as,A,5,,Y,H,0.09009
2,12as,A,6,,I,H,0.402367
3,12as,A,7,,A,H,0.462264
4,12as,A,8,,K,H,0.112195


In [30]:
import numpy as np
import pandas as pd

# Convert common NA strings to real missing values
labels_df["rsa"] = labels_df["rsa"].replace(["NA", "na", ""], np.nan)

# Force numeric (anything non-numeric becomes NaN)
labels_df["rsa"] = pd.to_numeric(labels_df["rsa"], errors="coerce")

# optional: ensure float type
labels_df["rsa"] = labels_df["rsa"].astype("float32")

labels_df["asa"].describe()

In [31]:
labels_df.to_parquet("data/processed/dssp_rsa_all_main.parquet", index=False)